# Case Study 110: Sports Strategy Analysis (Cricket - IPL)

## 1. Problem Definition
**Project Title:** IPL Match Strategy & Outcome Prediction

**Problem Statement:** A franchise wants to use historical IPL data to maximize their win probability. We will build and compare machine learning models to predict the winner based on purely pre-match strategic data.

**Objective:** Predict the match winner using pre-match data (teams, venue, toss) to establish a realistic baseline accuracy for sports prediction (typically 50-60%).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

import warnings
warnings.filterwarnings('ignore')

## 2. Dataset Loading
**Dataset:** `simple_matches.csv`
This is a streamlined dataset containing only the essential pre-match features.

In [ ]:
df = pd.read_csv('simple_matches.csv')
df = df.dropna()
df.head()

,team1,team2,venue,toss_winner,toss_decision,winner
0,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,Sunrisers Hyderabad
1,Mumbai Indians,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,field,Rising Pune Supergiant
2,Gujarat Lions,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,field,Kolkata Knight Riders
3,Rising Pune Supergiant,Kings XI Punjab,Holkar Cricket Stadium,Kings XI Punjab,field,Kings XI Punjab
4,Royal Challengers Bangalore,Delhi Daredevils,M Chinnaswamy Stadium,Royal Challengers Bangalore,bat,Royal Challengers Bangalore


## 3. Preprocessing & Encoding
We use a unified `LabelEncoder` for all team columns so the model recognizes team entities correctly.

In [ ]:
all_teams = pd.concat([df['team1'], df['team2'], df['toss_winner'], df['winner']]).unique()
le_team = LabelEncoder()
le_team.fit(all_teams)

le_venue = LabelEncoder()
le_toss_dec = LabelEncoder()

df['team1_enc'] = le_team.transform(df['team1'])
df['team2_enc'] = le_team.transform(df['team2'])
df['toss_winner_enc'] = le_team.transform(df['toss_winner'])
df['winner_enc'] = le_team.transform(df['winner'])
df['venue_enc'] = le_venue.fit_transform(df['venue'])
df['toss_decision_enc'] = le_toss_dec.fit_transform(df['toss_decision'])

## 4. Model Development (Comparing Meaningful Experiments)
We compare **Logistic Regression** vs **Random Forest** to find the most capable pre-match predictor.

In [ ]:
features = ['team1_enc', 'team2_enc', 'venue_enc', 'toss_winner_enc', 'toss_decision_enc']
X = df[features]
y = df['winner_enc']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Logistic Regression
log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train, y_train)
acc_log = accuracy_score(y_test, log_model.predict(X_test))

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf_model.predict(X_test))

print(f"Logistic Regression Accuracy: {acc_log*100:.2f}%")
print(f"Random Forest Accuracy: {acc_rf*100:.2f}%")

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_model.predict(X_test), target_names=le_team.classes_))

Logistic Regression Accuracy: 22.52%
Random Forest Accuracy: 51.66%

Random Forest Classification Report:
                             precision    recall  f1-score   support

        Chennai Super Kings       0.57      0.89      0.70        18
            Deccan Chargers       0.00      0.00      0.00         8
             Delhi Capitals       0.00      0.00      0.00         5
           Delhi Daredevils       0.38      0.38      0.38        13
              Gujarat Lions       0.00      0.00      0.00         2
            Kings XI Punjab       0.50      0.59      0.54        17
       Kochi Tuskers Kerala       0.00      0.00      0.00         1
      Kolkata Knight Riders       0.58      0.71      0.64        21
             Mumbai Indians       0.36      0.50      0.42        18
              Pune Warriors       0.00      0.00      0.00         1
           Rajasthan Royals       0.86      0.60      0.71        20
     Rising Pune Supergiant       0.00      0.00      0.00       

## 5. Model Export
The Random Forest model is exported for Streamlit deployment.

In [ ]:
joblib.dump(rf_model, 'cricket_model.pkl')
joblib.dump(le_team, 'le_team.pkl')
joblib.dump(le_venue, 'le_venue.pkl')
joblib.dump(le_toss_dec, 'le_toss_dec.pkl')

['le_toss_dec.pkl']